##  SETUPS ##


In [2]:
# Core libraries
import pandas as pd
import numpy as np
import re
from datetime import datetime

# The star of the show
from google_play_scraper import app, reviews, Sort

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [4]:
AWASH_APP_ID = 'com.sc.awashpay'

# Step 1: Get app metadata (rating, installs, description...)
app_info = app(
    AWASH_APP_ID,
    lang='en',    # Language: English
    country='et'  # Country: Ethiopia
)

print("=" * 50)
print("Awash Bank App Info")
print("=" * 50)
print(f"App Title   : {app_info['title']}")
print(f"Current Score: {app_info['score']}")
print(f"Total Ratings: {app_info['ratings']:,}")
print(f"Total Reviews: {app_info['reviews']:,}")
print(f"Installs     : {app_info['installs']}")

Awash Bank App Info
App Title   : AwashBIRR Pro
Current Score: 4.335774
Total Ratings: 18,844
Total Reviews: 3,415
Installs     : 1,000,000+


In [5]:
# Step 2: Scrape reviews
print(f"Scraping reviews for Awash Bank...")

result, continuation_token = reviews(
    AWASH_APP_ID,
    lang='en',
    country='et',
    sort=Sort.NEWEST,       # Most recent first
    count=400,              # Ask for more than 400 to be safe
    filter_score_with=None  # All star ratings
)

print(f"Collected {len(result)} raw reviews")

Scraping reviews for Awash Bank...
Collected 400 raw reviews


In [6]:
# Let's inspect what a single raw review looks like
print("Keys in a single review:")
print(list(result[0].keys()))

print("\nFirst raw review (sample):")
for key, value in result[0].items():
    print(f"  {key}: {value}")

Keys in a single review:
['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']

First raw review (sample):
  reviewId: c653753f-9d7b-4f1a-a433-cbf99190082c
  userName: Adem Beker
  userImage: https://play-lh.googleusercontent.com/a-/ALV-UjUyMaP1E-MdKL1e4fRPLH-byHVNpAODTxvgLT2YupEioB8xXgV4
  content: good app 👍
  score: 5
  thumbsUpCount: 1
  reviewCreatedVersion: 1.1.3
  at: 2026-05-15 08:00:26
  replyContent: None
  repliedAt: None
  appVersion: 1.1.3


In [7]:
# Step 3: Extract only the columns we need
raw_data = []

for r in result:
    raw_data.append({
        'review_id': r.get('reviewId', ''),
        'review'   : r.get('content', ''),
        'rating'   : r.get('score', None),
        'date'     : r.get('at', None),
        'bank'     : 'Awash Bank',
        'source'   : 'Google Play'
    })

# Build a DataFrame
df_raw = pd.DataFrame(raw_data)

print(f"Shape: {df_raw.shape}")
df_raw.head()

Shape: (400, 6)


,review_id,review,rating,date,bank,source
0,c653753f-9d7b-4f1a-a433-cbf99190082c,good app 👍,5,2026-05-15 08:00:26,Awash Bank,Google Play
1,fe879501-be94-4a3b-9e5d-4295b24d04e8,good,5,2026-05-14 18:48:44,Awash Bank,Google Play
2,4fbe211a-4c27-467f-8554-e59accc4233a,This app is the best to use. thank you! commen...,5,2026-05-13 14:06:56,Awash Bank,Google Play
3,26e6b3fe-7518-436d-8209-636361c867f8,tajaajila torban tokko osoo hinkenniin network...,2,2026-05-13 09:11:38,Awash Bank,Google Play
4,76ffa480-35a4-4ab8-bb1f-112c4864ff28,it's good,5,2026-05-12 15:29:21,Awash Bank,Google Play


In [8]:
# Basic shape and types
print(f"Total reviews collected: {len(df_raw)}")
print(f"\nColumn dtypes:")
print(df_raw.dtypes)

Total reviews collected: 400

Column dtypes:
review_id               str
review                  str
rating                int64
date         datetime64[us]
bank                    str
source                  str
dtype: object


In [9]:
# Rating distribution — what do users think?
print("Rating distribution:")
rating_counts = df_raw['rating'].value_counts().sort_index(ascending=False)
for rating, count in rating_counts.items():
    bar = '█' * (count // 5)
    print(f"  {int(rating)} stars: {count:>4}  {bar}")

Rating distribution:
  5 stars:  299  ███████████████████████████████████████████████████████████
  4 stars:   41  ████████
  3 stars:    9  █
  2 stars:   11  ██
  1 stars:   40  ████████


In [10]:
# What does the date column look like right now?
print("Sample date values (raw):")
print(df_raw['date'].head(10).to_string())

print(f"\nDate dtype: {df_raw['date'].dtype}")

Sample date values (raw):
0   2026-05-15 08:00:26
1   2026-05-14 18:48:44
2   2026-05-13 14:06:56
3   2026-05-13 09:11:38
4   2026-05-12 15:29:21
5   2026-05-10 12:51:30
6   2026-05-09 16:23:41
7   2026-05-09 04:47:57
8   2026-05-07 12:06:31
9   2026-05-07 01:23:04

Date dtype: datetime64[us]


## DATA QUALITY AUDIT ##

In [11]:
print("=" * 50)
print("DATA QUALITY AUDIT")
print("=" * 50)

# --- Problem 1: Missing Values ---
print("\nProblem 1: Missing Values")
print("-" * 30)
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)

for col in df_raw.columns:
    status = f"{missing[col]} missing ({missing_pct[col]}%)" if missing[col] > 0 else "OK"
    print(f"  {col:<15}: {status}")

DATA QUALITY AUDIT

Problem 1: Missing Values
------------------------------
  review_id      : OK
  review         : OK
  rating         : OK
  date           : OK
  bank           : OK
  source         : OK


In [12]:
# --- Problem 2: Duplicate Reviews ---
print("Problem 2: Duplicates")
print("-" * 30)

# Exact duplicates on review text
exact_dupes = df_raw.duplicated(subset=['review']).sum()
print(f"  Exact duplicate reviews : {exact_dupes}")

# Duplicate review IDs
id_dupes = df_raw.duplicated(subset=['review_id']).sum()
print(f"  Duplicate review IDs    : {id_dupes}")

# Empty reviews (also a form of bad data)
empty_reviews = (df_raw['review'].str.strip() == '').sum()
print(f"  Empty review texts      : {empty_reviews}")

Problem 2: Duplicates
------------------------------
  Exact duplicate reviews : 68
  Duplicate review IDs    : 0
  Empty review texts      : 0


In [13]:
# --- Problem 3: Date Format ---
print("Problem 3: Date Format")
print("-" * 30)
print(f"  Current dtype: {df_raw['date'].dtype}")
print(f"  Sample values: {df_raw['date'].iloc[0]}")
print(f"  Target format: YYYY-MM-DD (string or date object)")

Problem 3: Date Format
------------------------------
  Current dtype: datetime64[us]
  Sample values: 2026-05-15 08:00:26
  Target format: YYYY-MM-DD (string or date object)


## cleaning ##

In [27]:
# 1. Install the scraper library if you haven't already
# !pip install google-play-scraper

import pandas as pd
from google_play_scraper import Sort, reviews_all

# 2. Fetch ALL reviews for the Awash Bank mobile app
# 'com.awash.bank.mobile' is the official Google Play package ID for Awash
result = reviews_all(
    'com.sc.awashpay',
    lang='en', 
    country='us', 
    sort=Sort.NEWEST,
    count=400,
    filter_score_with=None 
)

 # All star ratings

print(f"Collected {len(result)} raw reviews")

# 3. Extract and format the data
raw_data = []
for r in result:
    raw_data.append({
        'review_id': r.get('reviewId', ''),
        'review'   : r.get('content', ''),
        'rating'   : r.get('score', None),
        'date'     : r.get('at', None),
        'bank'     : 'Awash Bank',
        'source'   : 'Google Play'
    })

# 4. Build the DataFrame and make your copy
df_raw = pd.DataFrame(raw_data)
df = df_raw.copy()
print(f"Starting with: {len(df)} reviews")




Collected 3380 raw reviews
Starting with: 3380 reviews


In [14]:
before = len(df)

# Drop rows missing the critical columns
critical_cols = ['review', 'rating']
df = df.dropna(subset=critical_cols)

removed = before - len(df)
print(f"Removed {removed} rows with missing critical data")
print(f"Remaining: {len(df)} reviews")

Removed 0 rows with missing critical data
Remaining: 3380 reviews


In [15]:
before = len(df)

df = df.drop_duplicates(subset=['review_id'], keep='first')

removed = before - len(df)
print(f"Removed {removed} duplicate reviews")
print(f"Remaining: {len(df)} reviews")

Removed 0 duplicate reviews
Remaining: 3380 reviews


In [16]:
print("Before normalization:")
print(df['date'].head(3).to_string())
print(f"dtype: {df['date'].dtype}")

# Convert to pandas datetime, then format as YYYY-MM-DD string
df['date'] = pd.to_datetime(df['date']).dt.strftime('%Y-%m-%d')

print("\nAfter normalization:")
print(df['date'].head(3).to_string())
print(f"dtype: {df['date'].dtype}")

print(f"\nDate range: {df['date'].min()} to {df['date'].max()}")

Before normalization:
0   2026-05-16 17:06:10
1   2026-05-16 13:54:35
2   2026-05-15 08:00:26
dtype: datetime64[us]

After normalization:
0    2026-05-16
1    2026-05-16
2    2026-05-15
dtype: str

Date range: 2023-11-07 to 2026-05-16


In [18]:
import re
def clean_text(text):
    """Standardize review text: collapse whitespace, strip edges."""
    if pd.isna(text):
        return ''
    text = str(text)
    text = re.sub(r'\s+', ' ', text)  # collapse multiple spaces/newlines
    text = text.strip()               # remove leading/trailing whitespace
    return text

# Show before/after on a sample review
sample_raw = "  Great   app!\n\nVery useful.  "
print(f"Before: {repr(sample_raw)}")
print(f"After : {repr(clean_text(sample_raw))}")

# Apply to the full column
df['review'] = df['review'].apply(clean_text)

# Remove any reviews that became empty after cleaning
before = len(df)
df = df[df['review'].str.len() > 0]
removed = before - len(df)
print(f"\nRemoved {removed} reviews that were empty after cleaning")

Before: '  Great   app!\n\nVery useful.  '
After : 'Great app! Very useful.'

Removed 0 reviews that were empty after cleaning


In [19]:
# Check for out-of-range ratings
invalid_ratings = df[(df['rating'] < 1) | (df['rating'] > 5)]
print(f"Invalid ratings (outside 1–5): {len(invalid_ratings)}")

# Remove them
df = df[(df['rating'] >= 1) & (df['rating'] <= 5)]

# Ensure rating is stored as integer
df['rating'] = df['rating'].astype(int)

print(f"Remaining: {len(df)} reviews")
print(f"Rating dtype: {df['rating'].dtype}")

Invalid ratings (outside 1–5): 0
Remaining: 3380 reviews
Rating dtype: int64


In [20]:
# Select only the 5 required columns in the right order
df_clean = df[['review', 'rating', 'date', 'bank', 'source']].copy()

# Sort by date (newest first) for clean presentation
df_clean = df_clean.sort_values('date', ascending=False).reset_index(drop=True)

print(f"Final dataset shape: {df_clean.shape}")
df_clean.head(10)

Final dataset shape: (3380, 5)


,review,rating,date,bank,source
0,but kobo town water bill payment is not included,5,2026-05-16,Awash Bank,Google Play
1,Bad application,1,2026-05-16,Awash Bank,Google Play
2,good app 👍,5,2026-05-15,Awash Bank,Google Play
3,good,5,2026-05-14,Awash Bank,Google Play
4,This app is the best to use. thank you! commen...,5,2026-05-13,Awash Bank,Google Play
5,tajaajila torban tokko osoo hinkenniin network...,2,2026-05-13,Awash Bank,Google Play
6,it's good,5,2026-05-12,Awash Bank,Google Play
7,happy,5,2026-05-10,Awash Bank,Google Play
8,nice,5,2026-05-09,Awash Bank,Google Play
9,Nice one,5,2026-05-09,Awash Bank,Google Play


In [21]:
# Save to CSV
import os
os.makedirs('data/processed', exist_ok=True)

output_path = 'data/processed/awash_bank_reviews_clean.csv'
df_clean.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

Saved to: data/processed/awash_bank_reviews_clean.csv
